# Canvas and grid

`bv.canvas` — one axes, several plots on it.
`bv.grid` — several panels, with ratios, sharing, spacing and spanning.

Both are context managers that save/show the figure when the block exits.

In [ ]:
import numpy as np
import behaviz as bv

x = np.linspace(1, 10, 80)
y = np.sin(x)
z = np.cos(x)

## 1. `bv.canvas` — one shared axes

Plot calls inside the block omit `ax=`: they auto-target the canvas.

In [ ]:
with bv.canvas() as ax:
    bv.plot_line(x, y, color="#787878")
    bv.plot_scatter(x, z, color="#991122")

Pass a spec for styling, and `save=` to write the figure on exit.

In [ ]:
spec = bv.PlotSpec(
    title="Canvas",
    x=bv.AxisSpec(label="Time", unit="s"),
    y=bv.AxisSpec(label="Signal"),
)

with bv.canvas(spec) as ax:
    bv.plot_line(x, y)
    bv.plot_line(x, z)

## 2. `bv.grid` — several panels

`nrows`/`ncols` yields a 2-D array indexed `axs[row, col]`.

Unlike `canvas` there is no single default target, so **every plot call needs `ax=`**.

In [ ]:
with bv.grid(2, 2) as axs:
    bv.plot_line(x, y, ax=axs[0, 0])
    bv.plot_scatter(x, z, ax=axs[0, 1])
    bv.plot_line(x, x**2, ax=axs[1, 0])
    bv.plot_scatter(x, np.sqrt(x), ax=axs[1, 1])

### Ratios, spacing, sharing, figure title

- `width_ratios` / `height_ratios` — relative column widths / row heights
- `wspace` / `hspace` — gaps between panels
- `sharex` / `sharey` — link the panel axes
- `suptitle` — figure-level title

In [ ]:
with bv.grid(
    2, 2,
    width_ratios=[2, 1],
    height_ratios=[1, 2],
    wspace=0.3, hspace=0.3,
    sharex=True,
    suptitle="Ratios + shared x",
) as axs:
    bv.plot_line(x, y, ax=axs[0, 0])
    bv.plot_line(x, z, ax=axs[0, 1])
    bv.plot_scatter(x, x**2, ax=axs[1, 0])
    bv.plot_scatter(x, np.sqrt(x), ax=axs[1, 1])

## 3. Spanning panels with a mosaic

A mosaic is ASCII art: one character per cell, one line per row.
Repeat a character to make that panel **span** those cells.

```
AAB
CCB
```
`A` spans the top two columns, `C` the bottom two, `B` spans both rows on the right.

With `mosaic=` you get a **dict** keyed by panel name instead of an array.

In [ ]:
with bv.grid(mosaic="AAB\nCCB", suptitle="AAB / CCB") as axs:
    bv.plot_line(x, x**2, ax=axs["A"])
    bv.plot_line(x, y, ax=axs["B"])
    bv.plot_scatter(x, np.sqrt(x), ax=axs["C"])

Triple-quoted strings read more like the layout they describe:

In [ ]:
layout = """
AAA
BCC
BCC
"""

with bv.grid(mosaic=layout) as axs:
    bv.plot_line(x, y, ax=axs["A"])       # full-width banner
    bv.plot_scatter(x, z, ax=axs["B"])    # tall left panel
    bv.plot_line(x, x**2, ax=axs["C"])    # large 2x2 block

### Empty cells

`.` leaves a cell blank — useful for staircase / triangular layouts.

In [ ]:
with bv.grid(mosaic="AB\n.C") as axs:
    bv.plot_line(x, y, ax=axs["A"])
    bv.plot_line(x, z, ax=axs["B"])
    bv.plot_scatter(x, x**2, ax=axs["C"])

Each panel name must form a **solid rectangle**. An L-shape is rejected:

```python
bv.grid(mosaic="ABA\nCCC")   # ValueError: panel 'A' is not a solid rectangle
```

## 4. Per-panel specs

The grid-level `spec` sets the figure (size, dpi, style). Each panel takes its
own `spec=` on the plot call.

In [ ]:
left  = bv.PlotSpec(title="Linear", x=bv.AxisSpec(label="x"), y=bv.AxisSpec(label="y"))
right = bv.PlotSpec(title="Log x",  x=bv.AxisSpec(label="x", scale="log"), y=bv.AxisSpec(label="y"))

with bv.grid(1, 2, spec=bv.PlotSpec(figure=bv.FigureSpec(figsize=(10, 4)))) as axs:
    bv.plot_line(x, x**2, ax=axs[0, 0], spec=left)
    bv.plot_line(x, x**2, ax=axs[0, 1], spec=right)

## 5. Backends

The same grid code runs on every backend — only `bv.set_renderer` changes.

In [ ]:
from bokeh.io import show,output_notebook
output_notebook()

In [ ]:
for backend in ["matplotlib", "seaborn", "bokeh"]:
    bv.set_renderer(backend)
    with bv.grid(mosaic="AAB\nCCB", save=f"grid_{backend}." + ("html" if backend == "bokeh" else "png"), show=True) as axs:
        bv.plot_line(x, x**2, ax=axs["A"])
        bv.plot_line(x, y, ax=axs["B"])
        bv.plot_scatter(x, np.sqrt(x), ax=axs["C"])
    print(backend, "ok")

bv.set_renderer("matplotlib")

Bokeh writes interactive `.html`; matplotlib/seaborn write `.png`/`.pdf`/`.svg`.
Spanning, ratios, sharing and spacing work on all three.

## 6. Matplotlib-only extras

Three features have no bokeh equivalent. Rather than degrade silently, bokeh
raises `NotImplementedError`.

In [ ]:
# layout engine — "tight" or "constrained"
with bv.grid(1, 2, layout="constrained") as axs:
    bv.plot_line(x, y, ax=axs[0, 0])
    bv.plot_line(x, z, ax=axs[0, 1])

In [ ]:
# inset axes — bounds are (x, y, width, height) in parent-axes fractions
with bv.grid(1, 1) as axs:
    bv.plot_line(x, x**2, ax=axs[0, 0])
    ins = bv.inset(axs[0, 0], (0.6, 0.15, 0.35, 0.3))
    bv.plot_line(x, y, ax=ins)

In [ ]:
# one colorbar shared by several panels
img = np.random.rand(12, 12)

with bv.grid(1, 2) as axs:
    for c in range(2):
        fig, ax = bv.plot_image(img, ax=axs[0, c])
    bv.shared_colorbar(fig, axs, mappable=ax.images[-1])

On bokeh these raise instead of producing a wrong figure:

In [ ]:
bv.set_renderer("bokeh")
try:
    with bv.grid(1, 2, layout="tight") as axs:
        pass
except NotImplementedError as e:
    print("NotImplementedError:", e)

bv.set_renderer("matplotlib")

## Summary

| | `bv.canvas` | `bv.grid` |
|---|---|---|
| Panels | one | many |
| `ax=` on plot calls | optional | **required** |
| Yields | the axes | `axs[r, c]` array, or `axs["A"]` dict with `mosaic=` |

`grid` options: `nrows`/`ncols` or `mosaic`, `width_ratios`, `height_ratios`,
`wspace`, `hspace`, `sharex`, `sharey`, `suptitle`, `layout`, `spec`, `save`, `show`.

matplotlib/seaborn only: `layout=`, `bv.inset`, `bv.shared_colorbar`.